In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('/Users/krishnag02/Learning Phase/Jupyterfiles/food_delivery_orders_dataset.csv')

In [24]:
df.head()

,order_id,customer_id,restaurant_id,driver_id,order_timestamp,order_date,order_hour,day_of_week,is_weekend,city,...,traffic_level,delivery_partner_experience_months,delivery_partner_rating,restaurant_preparation_time_minutes,estimated_delivery_time_minutes,actual_delivery_time_minutes,late_delivery,order_status,cancellation_reason,customer_rating
0,ORD_000001,CUS_00001,RES_0008,DRV_0075,2025-10-05 15:54:00,2025-10-05,15,Sunday,1,City_C,...,Medium,14,4.4,25,25,29.0,0.0,Completed,NaN,4.6
1,ORD_000002,CUS_00013,RES_0024,DRV_0131,2025-10-01 20:00:00,2025-10-01,20,Wednesday,0,City_A,...,High,44,4.7,12,20,24.0,0.0,Completed,NaN,5.0
2,ORD_000003,CUS_00049,RES_0013,DRV_0087,2025-10-07 18:24:00,2025-10-07,18,Tuesday,0,City_C,...,High,1,5.0,11,63,72.0,1.0,Completed,NaN,3.6
3,ORD_000004,CUS_00007,RES_0001,DRV_0177,2025-10-30 16:54:00,2025-10-30,16,Thursday,0,City_A,...,Low,50,4.3,12,23,31.0,1.0,Completed,NaN,4.1
4,ORD_000005,CUS_00325,RES_0029,DRV_0114,2025-10-15 18:34:00,2025-10-15,18,Wednesday,0,City_B,...,Medium,21,4.0,9,32,42.0,1.0,Completed,NaN,3.9


# 🧑‍💼 Client Requirement #1 — Business Overview

We have recently collected our food-delivery order data and want to understand the overall health of the platform before we start deeper analysis.

Please provide me with the following:

- How many total orders do we have?
- How many unique customers placed orders?
- How many unique restaurants are present?
- What percentage of orders were successfully completed?
- What percentage of orders were cancelled?

Please return the results in a clean summary format.

In [102]:
total_orders = df.shape[0]
Customers = df['customer_id'].nunique()
Restaurants = df['restaurant_id'].nunique()
Percentage_completed = (((df.groupby('order_status')['order_id'].size()['Completed'])/df.shape[0])*100)
Percentage_cancelled = (((df.groupby('order_status')['order_id'].size()['Cancelled'])/df.shape[0])*100)

In [107]:
print(f'Total Orders : {total_orders}\nNo of Customers : {Customers}\nNo of Restaurants : {Restaurants}\nCompleted_orders: {Percentage_completed}%\nCancelled_order : {Percentage_cancelled}%')

Total Orders : 50000
No of Customers : 4979
No of Restaurants : 250
Completed_orders: 98.264%
Cancelled_order : 1.736%


# 🧑‍💼 Client Requirement #2 — Customer Behavior

Thanks. The overall cancellation rate looks low, but I want to understand which customers are contributing most to our order volume.

For every customer, calculate:

- Total number of orders
- Number of completed orders
- Number of cancelled orders
- Total amount spent
- Average order value

Then identify the top 10 customers by total amount spent.

One important business rule:

Cancelled orders should NOT contribute to customer spending.

Return the result sorted from highest to lowest total spending.

In [ ]:
temp = df.groupby(['customer_id','order_status'])[['order_id','order_total']].agg({'order_id':'size','order_total':['sum','mean']}).unstack()
temp.fillna(0,inplace=True)

In [109]:
client = pd.DataFrame()
client['Total_order'] = (temp['order_id']['size']['Cancelled']) + (temp['order_id']['size']['Completed'])
client['Completed_orders'] = temp['order_id']['size']['Completed']
client['Cancelled_orders'] = temp['order_id']['size']['Cancelled']
client['Total_spent'] = temp['order_total']['sum']['Completed']
client['Avg_order_value'] = temp['order_total']['mean']['Completed']
client.sort_values('Total_spent',ascending=False).head(10)

,Total_order,Completed_orders,Cancelled_orders,Total_spent,Avg_order_value
customer_id,,,,,
CUS_00001,4510.0,4445.0,65.0,146326.36,32.919316
CUS_00003,2138.0,2102.0,36.0,91792.49,43.669120
CUS_00004,2319.0,2282.0,37.0,84788.20,37.155215
CUS_00002,2065.0,2021.0,44.0,67110.38,33.206522
CUS_00005,1506.0,1483.0,23.0,61700.51,41.605199
CUS_00006,1297.0,1277.0,20.0,52645.75,41.226116
CUS_00007,1214.0,1189.0,25.0,43895.30,36.917830
CUS_00008,1102.0,1076.0,26.0,35322.44,32.827546
CUS_00009,823.0,817.0,6.0,34223.30,41.888984


# 🧑‍💼 Client Requirement #3 — Restaurant Performance

Now let's move to the restaurant side.

The business team wants to identify which restaurants are performing best, but they don't want cancellations to inflate revenue.

For each restaurant, calculate:

- total_orders
- completed_orders
- cancelled_orders
- total_revenue — Completed orders only
- avg_order_value — Completed orders only
- avg_customer_rating — Completed orders only

Then answer:

What are the top 10 restaurants by total revenue?

In [120]:
temp = df.groupby(['restaurant_id','order_status'])[['order_id','order_total','customer_rating']].agg({'order_id' : 'size', 'order_total' : ['sum','mean'], 'customer_rating' : 'mean'}).unstack()
temp.fillna(0,inplace=True)

order_id           order_total                         \
                   size                   sum                   mean   
order_status  Cancelled Completed   Cancelled  Completed   Cancelled   
restaurant_id                                                          
RES_0001          162.0    8675.0     5557.29  286626.46   34.304259   
RES_0002           52.0    3501.0     1755.16  120102.18   33.753077   
RES_0003           64.0    3903.0     1570.62   81038.98   24.540937   
RES_0004           62.0    4199.0     2159.26  141270.79   34.826774   
RES_0005           20.0    1344.0      604.81   44809.81   30.240500   
...                 ...       ...         ...        ...         ...   
RES_0246            0.0      14.0        0.00     569.34    0.000000   
RES_0247            0.0      48.0        0.00    2444.45    0.000000   
RES_0248            0.0      13.0        0.00     426.13    0.000000   
RES_0249            0.0      21.0        0.00    2490.29    0.000000   
RES_0250            1.0      28.0      223.83    2652.30  223.830000   

                          customer_rating            
                                     mean            
order_status    Completed       Cancelled Completed  
restaurant_id                                        
RES_0001        33.040514             0.0  4.286110  
RES_0002        34.305107             0.0  4.308312  
RES_0003        20.763254             0.0  4.340584  
RES_0004        33.643913             0.0  4.275923  
RES_0005        33.340632             0.0  4.291815  
...                   ...             ...       ...  
RES_0246        40.667143             0.0  4.135714  
RES_0247        50.926042             0.0  4.435417  
RES_0248        32.779231             0.0  4.330769  
RES_0249       118.585238             0.0  4.300000  
RES_0250        94.725000             0.0  4.478571  

[250 rows x 8 columns]

In [125]:
client3 = pd.DataFrame(index=temp.index)
client3['Total_orders'] = (temp['order_id']['size']['Cancelled'] + temp['order_id']['size']['Completed'])
client3['Completed_orders'] = temp['order_id']['size']['Completed']
client3['Cancelled_orders'] = temp['order_id']['size']['Cancelled']
client3['Total_revenue'] = temp['order_total']['sum']['Completed']
client3['Avg_order_value'] = temp['order_total']['mean']['Completed']
client3['Avg_customer_rating'] = temp['customer_rating']['mean']['Completed']
client3.sort_values('Total_revenue', ascending = False).head(10)

,Total_orders,Completed_orders,Cancelled_orders,Total_revenue,Avg_order_value,Avg_customer_rating
restaurant_id,,,,,,
RES_0001,8837.0,8675.0,162.0,286626.46,33.040514,4.286110
RES_0004,4261.0,4199.0,62.0,141270.79,33.643913,4.275923
RES_0002,3553.0,3501.0,52.0,120102.18,34.305107,4.308312
RES_0009,1621.0,1595.0,26.0,82140.23,51.498577,4.281693
RES_0003,3967.0,3903.0,64.0,81038.98,20.763254,4.340584
RES_0008,1265.0,1247.0,18.0,62351.98,50.001588,4.340176
RES_0006,1141.0,1127.0,14.0,58309.62,51.738793,4.292990
RES_0012,466.0,458.0,8.0,46851.94,102.296812,4.315721
RES_0005,1364.0,1344.0,20.0,44809.81,33.340632,4.291815


# 🧑‍💼 Client Requirement #4 — City Performance

Now management wants to understand which cities are driving the business.

For each city, calculate:

- total_orders
- completed_orders
- cancelled_orders
- total_revenue — Completed orders only
- avg_order_value — Completed orders only
- cancellation_rate — percentage of all orders that were cancelled
- 
avg_delivery_time — Completed orders only

Then answer:

Which 5 cities generate the highest total revenue?

Important business rules
Revenue → Completed only
AOV → Completed only
Delivery time → Completed only
Cancellation rate → Cancelled / Total orders × 100

In [129]:
temp = df.groupby(['city','order_status'])[['order_id','order_total']].agg({'order_id':'size', 'order_total':['sum','mean']})
temp.fillna(0,inplace = True).unstack()

order_id           order_total                                 
                  size                   sum                  mean           
order_status Cancelled Completed   Cancelled  Completed  Cancelled  Completed
city                                                                         
City_A             361     19527    13530.13  717733.52  37.479584  36.755954
City_B             258     14627    10246.89  601434.09  39.716628  41.118075
City_C             172     10006     6921.53  340471.27  40.241453  34.026711
City_D              77      4972     3599.93  214537.69  46.752338  43.149173

In [130]:
client4 = pd.DataFrame(index = temp.index) 
client4['Total_orders'] = (temp['order_id']['size']['Cancelled'] + temp['order_id']['size']['Completed'])
client4['Completed_orders'] = temp['order_id']['size']['Completed']
client4['Cancelled_orders'] = temp['order_id']['size']['Cancelled']
client4['Total_revenue'] = temp['order_total']['sum']['Completed']
client4['Avg_order_value'] = temp['order_total']['mean']['Completed']
client4['Cancellation_rate'] = (client4['Cancelled_orders']/client4['Total_orders'])*100

KeyError: 'Cancelled'